In [2]:
import os

os.environ["OPENAI_API_KEY"] = "sk-*****************"
os.environ["OPENAI_BASE_URL"] = "https://dashscope.aliyuncs.com/compatible-mode/v1"
os.environ["OPENAI_MODEL"] = "qwen-turbo"

In [34]:
from typing import List
from langchain_core.messages import BaseMessage
from langchain_openai.chat_models.base import BaseChatOpenAI

class ChildChatOpenAI(ChatOpenAI):
    def get_num_tokens_from_messages(self, messages: List[BaseMessage]) -> int:
        model, encoding = self._get_encoding_model()
        # 注意：这里只是解决了计算的问题，具体计算方案具体准确取决于具体模型
        if model.startswith("qwen-turbo"):
            # 调用祖父类的函数
            return super(BaseChatOpenAI, self).get_num_tokens_from_messages(messages)
        else:
            return super().get_num_tokens_from_messages(messages)

In [35]:
llm = ChildChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),  # 若走原生 OpenAI，可传 None/不传
    temperature=0.2,                        # 稳定输出
    timeout=1200,                             # 超时保护（秒）
    max_retries=2                           # 简单重试
)

In [6]:
pip install -U langchain-community

Note: you may need to restart the kernel to use updated packages.


In [7]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain.memory import ChatMessageHistory

# 1. 初始化一个空的记忆历史对象
history = ChatMessageHistory()

# 2. 添加一条系统消息，为AI设定角色
history.add_message(SystemMessage(content="你是一个有用的AI助手，请用中文回答问题。"))
print("✓ 添加系统消息")

# 3. 使用便捷方法添加用户消息
history.add_user_message("你好，我是小明")
print("✓ 添加用户消息: 你好，我是小明")

# 4. 使用便捷方法添加AI消息
history.add_ai_message("你好小明！很高兴认识你。有什么我可以帮助你的吗？")
print("✓ 添加AI消息: 你好小明！很高兴认识你。...")

# 5. 查看当前存储的所有消息
print("\n--- 当前消息历史 ---")
print(history.messages)

✓ 添加系统消息
✓ 添加用户消息: 你好，我是小明
✓ 添加AI消息: 你好小明！很高兴认识你。...

--- 当前消息历史 ---
[SystemMessage(content='你是一个有用的AI助手，请用中文回答问题。', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，我是小明', additional_kwargs={}, response_metadata={}), AIMessage(content='你好小明！很高兴认识你。有什么我可以帮助你的吗？', additional_kwargs={}, response_metadata={})]


In [8]:
# --- 准备工作 ---
history = ChatMessageHistory()
history.add_message(SystemMessage(content="你是一个有用的AI助手。请记住用户告诉你的信息，并在后续对话中使用。"))

# --- 第一轮对话：用户提供信息 ---
user_input1 = "我叫张三，今年25岁，是一名软件工程师"
history.add_user_message(user_input1)

# 调用LLM，传入完整历史
response1 = llm.invoke(history.messages) 
ai_response1 = response1.content
history.add_ai_message(ai_response1)

print(f"用户: {user_input1}")
print(f"AI: {ai_response1}\n")

# --- 第二轮对话：测试记忆 ---
user_input2 = "我的职业是什么？"
history.add_user_message(user_input2)

# 再次调用LLM，传入【更新后】的完整历史
response2 = llm.invoke(history.messages)
ai_response2 = response2.content
history.add_ai_message(ai_response2)

print(f"用户: {user_input2}")
print(f"AI: {ai_response2}\n")

# --- 查看最终的对话历史 ---
print("--- 最终消息历史 ---")
for msg in history.messages:
    print(f"[{type(msg).__name__}] {msg.content}")

用户: 我叫张三，今年25岁，是一名软件工程师
AI: 你好，张三！很高兴认识你。作为一名软件工程师，你的工作一定很有趣吧？我非常乐意了解你的想法和经历。有什么我可以帮你的吗？

用户: 我的职业是什么？
AI: 你的职业是软件工程师。你之前提到过，你今年25岁，是一名软件工程师。如果你有任何与工作相关的问题或需要帮助，我随时在这里为你提供支持！

--- 最终消息历史 ---
[SystemMessage] 你是一个有用的AI助手。请记住用户告诉你的信息，并在后续对话中使用。
[HumanMessage] 我叫张三，今年25岁，是一名软件工程师
[AIMessage] 你好，张三！很高兴认识你。作为一名软件工程师，你的工作一定很有趣吧？我非常乐意了解你的想法和经历。有什么我可以帮你的吗？
[HumanMessage] 我的职业是什么？
[AIMessage] 你的职业是软件工程师。你之前提到过，你今年25岁，是一名软件工程师。如果你有任何与工作相关的问题或需要帮助，我随时在这里为你提供支持！


In [9]:
history = ChatMessageHistory()
history.add_message(SystemMessage(content="你是一个有用的助手"))
max_messages = 6 # 设置一个较小的最大消息数（包括系统消息）

for i in range(5):
    history.add_user_message(f"用户消息 {i+1}")
    history.add_ai_message(f"AI回复 {i+1}")
    
    # 检查并限制消息数量
    if len(history.messages) > max_messages:
        print(f"  消息数达到 {len(history.messages)}，超过限制 {max_messages}，开始截断...")
        # 保留系统消息和最近的消息
        system_messages = [msg for msg in history.messages if isinstance(msg, SystemMessage)]
        dialogue_messages = [msg for msg in history.messages if not isinstance(msg, SystemMessage)]
        
        # 保留最近的 (max_messages - system_messages数量) 条对话消息
        history.messages = system_messages + dialogue_messages[-(max_messages - len(system_messages)):]
        print(f"  截断后消息数为: {len(history.messages)}")

print("\n--- 最终消息状态 ---")
for msg in history.messages:
    print(f"[{type(msg).__name__.replace('Message','')}] {msg.content}")

  消息数达到 7，超过限制 6，开始截断...
  截断后消息数为: 6
  消息数达到 8，超过限制 6，开始截断...
  截断后消息数为: 6
  消息数达到 8，超过限制 6，开始截断...
  截断后消息数为: 6

--- 最终消息状态 ---
[System] 你是一个有用的助手
[AI] AI回复 3
[Human] 用户消息 4
[AI] AI回复 4
[Human] 用户消息 5
[AI] AI回复 5


In [12]:
def simple_token_count(messages):
    # 更准确的中文token计数方法：将每个中文字符视为一个token
    # 注意：这仍然是简化方法，实际LLM的tokenization更复杂
    return sum(len(msg.content) for msg in messages)

history = ChatMessageHistory()
max_tokens = 50  # 设置一个较小的最大Token数

messages_to_add = [
    ("user", "你好，我想了解人工智能的发展历史"),
    ("ai", "人工智能起源于20世纪50年代，当时科学家们开始探索让机器模拟人类智能的可能性"),
    ("user", "那么机器学习和深度学习是什么时候发展起来的？"),
    ("ai", "机器学习在20世纪80年代开始兴起，而深度学习则在2010年代迎来了突破性发展"),
]

for msg_type, content in messages_to_add:
    if msg_type == "user":
        history.add_user_message(content)
    else:
        history.add_ai_message(content)
    
    token_count = simple_token_count(history.messages)
    print(f"添加消息后: Token数 = {token_count}")
    
    while token_count > max_tokens and len(history.messages) > 1:
        removed_msg = history.messages.pop(0)  # 移除最旧的消息
        token_count = simple_token_count(history.messages)
        print(f"  Token超限，移除一条旧消息。当前Token数: {token_count}")

print("\n--- 最终消息状态 (Token数: {}) ---".format(simple_token_count(history.messages)))
for msg in history.messages:
    print(f"[{type(msg).__name__.replace('Message','')}] {msg.content}")

添加消息后: Token数 = 16
添加消息后: Token数 = 55
  Token超限，移除一条旧消息。当前Token数: 39
添加消息后: Token数 = 61
  Token超限，移除一条旧消息。当前Token数: 22
添加消息后: Token数 = 61
  Token超限，移除一条旧消息。当前Token数: 39

--- 最终消息状态 (Token数: 39) ---
[AI] 机器学习在20世纪80年代开始兴起，而深度学习则在2010年代迎来了突破性发展


In [28]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferWindowMemory

# 1. 初始化一个窗口记忆，只保留最近 k=2 轮对话
# memory_window = ConversationBufferWindowMemory(k=2)
memory_window = ConversationBufferWindowMemory(k=2) # 为了演示效果更明显，设为1

# 2. 创建一个对话链，并将记忆对象传入
conversation = ConversationChain(
    llm=llm,
    memory=memory_window,
    verbose=True # 设置为True可以看到链的思考过程
)

# --- 模拟对话 ---
response1 = conversation.predict(input="我叫李四，住在上海。")
print(response1)

response2 = conversation.predict(input="我喜欢编程。")
print(response2)

response3 = conversation.predict(input="我叫什么名字？")
print(response3)

response4 = conversation.predict(input="我住在哪里？")
print(response4)

# --- 查看记忆内容 ---
print(memory_window.load_memory_variables({}))



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: 我叫李四，住在上海。
AI:

> Finished chain.
很高兴认识你，李四！上海是一个非常美丽且充满活力的城市，有外滩的夜景、南京路的繁华，还有许多美食和文化景点。你平时在上海喜欢做什么呢？或者有什么特别想了解的事情吗？我很乐意和你分享更多细节！


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: 我叫李四，住在上海。
AI: 很高兴认识你，李四！上海是一个非常美丽且充满活力的城市，有外滩的夜景、南京路的繁华，还有许多美食和文化景点。你平时在上海喜欢做什么呢？或者有什么特别想了解的事情吗？我很乐意和你分享更多细节！
Human: 我喜欢编程。
AI:

> Finished chain.
哦，你喜欢编程啊！那真是太棒了！编程是一门非常有趣且充满创造力的技能，可以用来解决各种问题，甚至改变世界。上海作为中国科技

In [36]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chains import ConversationChain


# 使用一个模拟的LLM（返回预设响应）
responses = [
    "巴赫确实是一位伟大的作曲家。",
    "听起来很健康！",
    "黄山以其奇松、怪石、云海而闻名。",
    "那一定是一次难忘的经历。",
    "你们讨论了古典音乐、爬山和黄山之旅。"
]

# 初始化记忆
memory = ConversationSummaryBufferMemory(
    llm=llm, 
    max_token_limit=100,
    return_messages=True
)

# 添加对话
memory.save_context(
    {"input": "我喜欢古典音乐，特别是巴赫的作品。"}, 
    {"output": "巴赫确实是一位伟大的作曲家。"}
)
memory.save_context(
    {"input": "我还喜欢在周末去爬山。"}, 
    {"output": "听起来很健康！"}
)
memory.save_context(
    {"input": "上周末我去了黄山，风景很美。"}, 
    {"output": "黄山以其奇松、怪石、云海而闻名。"}
)
memory.save_context(
    {"input": "是的，我还看到了日出。"}, 
    {"output": "那一定是一次难忘的经历。"}
)

# 查看记忆内容
memory_vars = memory.load_memory_variables({})
print("当前记忆内容:")
for msg in memory_vars['history']:
    print(f"{msg.type}: {msg.content}")

# 创建对话链
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False
)

# 测试对话
response = conversation.predict(input="你能总结一下我们刚才的对话吗？")
print("\nAI的回应:", response)

当前记忆内容:
system: The human says they like classical music, especially Bach's works. The AI acknowledges that Bach is a great composer. The human also mentions they enjoy hiking on weekends.
ai: 听起来很健康！
human: 上周末我去了黄山，风景很美。
ai: 黄山以其奇松、怪石、云海而闻名。
human: 是的，我还看到了日出。
ai: 那一定是一次难忘的经历。

AI的回应: 当然可以！我们刚才的对话内容如下：

1. 人类提到他们喜欢古典音乐，尤其是巴赫的作品。AI回应说巴赫是一位伟大的作曲家。
2. 人类还提到他们喜欢在周末去徒步旅行。AI回应说这听起来很健康。
3. 人类说他们上周末去了黄山，风景很美。AI回应说黄山以其奇松、怪石、云海而闻名。
4. 人类补充说他们还看到了日出。AI回应说那一定是一次难忘的经历。

总的来说，我们的对话围绕着音乐爱好和一次黄山之旅展开，AI对这些话题都做出了相应的回应。


In [39]:
import os
from langchain.memory import ConversationEntityMemory
from langchain.chains import ConversationChain
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

# 1. 初始化实体记忆
memory = ConversationEntityMemory(llm=llm)

# 2. 创建自定义的PromptTemplate
template = """以下是对话历史和一些实体信息:

实体信息:
{entities}

对话历史:
{history}

当前输入: {input}
请根据以上信息进行回复:"""

prompt = PromptTemplate(
    input_variables=["entities", "history", "input"],
    template=template
)

# 3. 创建对话链
conversation = ConversationChain(
    llm=llm,
    prompt=prompt,
    memory=memory,
    verbose=True
)

# --- 模拟对话 ---
print("=== 第一轮对话 ===")
response1 = conversation.predict(input="我的朋友小王在一家叫'InnovateTech'的公司工作。")
print(f"AI: {response1}\n")

print("=== 第二轮对话 ===")
response2 = conversation.predict(input="InnovateTech是一家做人工智能的初创公司。")
print(f"AI: {response2}\n")

print("=== 第三轮对话 ===")
response3 = conversation.predict(input="小王最近怎么样？")
print(f"AI: {response3}\n")

# --- 查看记忆内容 ---
print("=== 记忆内容 ===")
# 修复：传递一个包含输入键的字典
memory_vars = memory.load_memory_variables({"input": "查看记忆内容"})
print("对话历史:")
print(memory_vars['history'])
print("\n实体记忆:")
print(memory_vars['entities'])

# --- 测试实体记忆的使用 ---
print("\n=== 测试实体记忆的使用 ===")
response4 = conversation.predict(input="InnovateTech是做什么的？")
print(f"AI: {response4}")

=== 第一轮对话 ===


> Entering new ConversationChain chain...
Prompt after formatting:
以下是对话历史和一些实体信息:

实体信息:
{'小王': '', 'InnovateTech': ''}

对话历史:


当前输入: 我的朋友小王在一家叫'InnovateTech'的公司工作。
请根据以上信息进行回复:

> Finished chain.
AI: 很高兴认识你的朋友小王，他在InnovateTech工作一定很忙碌吧？如果你有其他想聊的内容，随时告诉我！

=== 第二轮对话 ===


> Entering new ConversationChain chain...
Prompt after formatting:
以下是对话历史和一些实体信息:

实体信息:
{'InnovateTech': "InnovateTech is a company where the human's friend Xiao Wang works."}

对话历史:
Human: 我的朋友小王在一家叫'InnovateTech'的公司工作。
AI: 很高兴认识你的朋友小王，他在InnovateTech工作一定很忙碌吧？如果你有其他想聊的内容，随时告诉我！

当前输入: InnovateTech是一家做人工智能的初创公司。
请根据以上信息进行回复:

> Finished chain.
AI: 很高兴你提到InnovateTech是一家做人工智能的初创公司！听起来小王在一家充满创新和活力的公司工作，一定有很多有趣的机会和挑战。如果你对人工智能或者小王的工作内容感兴趣，可以继续和我聊聊！

=== 第三轮对话 ===


> Entering new ConversationChain chain...
Prompt after formatting:
以下是对话历史和一些实体信息:

实体信息:
{}

对话历史:
Human: 我的朋友小王在一家叫'InnovateTech'的公司工作。
AI: 很高兴认识你的朋友小王，他在InnovateTech工作一定很忙碌吧？如果你有其他想聊的内容，随时告诉我！
Human: InnovateTech是一家做人工智能的初创公司。
AI: 很高兴你提到Inno

In [40]:
!pip install -qU langchain-redis redis

In [45]:
from langchain_community.chat_message_histories import RedisChatMessageHistory

# 2. 为不同会话定义唯一的 session_id
session_id_user_A = "user_session_A_123"
session_id_user_B = "user_session_B_456"

# 3. 初始化指向 Redis 的历史记录对象
history_A = RedisChatMessageHistory(
    session_id=session_id_user_A, 
    url="redis://localhost:6379/0"
)
history_B = RedisChatMessageHistory(
    session_id=session_id_user_B, 
    url="redis://localhost:6379/0"
)
# 4. 向会话A添加内容
history_A.add_user_message("我喜欢喝咖啡")
history_A.add_ai_message("知道了，你喜欢喝咖啡。")

# 5. 向会话B添加内容
history_B.add_user_message("我是个程序员")
history_B.add_ai_message("很棒！编程是一个很有趣的职业。")

# 6. 验证隔离性
print(f"会话A的历史: {[msg.content for msg in history_A.messages]}")
print(f"会话B的历史: {[msg.content for msg in history_B.messages]}")

# 7. 模拟应用重启后，重新加载会话A
print("\n--- 模拟应用重启 ---")
reloaded_history_A = RedisChatMessageHistory(
    session_id=session_id_user_A, 
    url="redis://localhost:6379/0"  # 只在这里指定 URL
)
print(f"重新加载的会话A历史: {[msg.content for msg in reloaded_history_A.messages]}")

会话A的历史: ['我喜欢喝咖啡', '知道了，你喜欢喝咖啡。']
会话B的历史: ['我是个程序员', '很棒！编程是一个很有趣的职业。']

--- 模拟应用重启 ---
重新加载的会话A历史: ['我喜欢喝咖啡', '知道了，你喜欢喝咖啡。']


In [46]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. 定义一个包含历史记录占位符的LCEL链
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])
chain = prompt | llm

# 2. 准备一个存储，用于根据session_id管理多个history对象
store = {}
def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# 3. 使用RunnableWithMessageHistory包装链
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# 4. 调用时，传入config指定session_id
config = {"configurable": {"session_id": "my_session_1"}}

# --- 模拟对话 ---
response1 = with_message_history.invoke({"input": "Hi! I'm Bob"}, config=config)
print(response1)

response2 = with_message_history.invoke({"input": "What's my name?"}, config=config)
print(response2)


# 5. 检查底层存储
print("--- 底层存储内容 ---")
print(store["my_session_1"].messages)

13:05:51 httpx INFO   HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
content='Hi, Bob! Nice to meet you. How can I assist you today? 😊' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 28, 'total_tokens': 46, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'qwen-turbo', 'system_fingerprint': None, 'id': 'chatcmpl-a952c095-4d81-9153-b762-d7aafd6f6830', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None} id='run--ca6a9359-7586-45db-87af-089d857f913d-0' usage_metadata={'input_tokens': 28, 'output_tokens': 18, 'total_tokens': 46, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}
13:05:52 httpx INFO   HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
content='Hi, Bob! 😊 You told me your name is Bob. How can I help yo